# Valence Band Offset (VBO), Bulk Method

Calculate the valence band offset of a periodic superlattice of two materials with the bulk method, using a DFT workflow on the Mat3ra platform.

The valence band maximum of each material is measured from the macroscopically averaged electrostatic potential (ESP) of its own bulk cell, and the superlattice gives the step in the averaged ESP between the centres of the two blocks:

$$\Delta E_v = \left[E_{\mathrm{VBM}} - \bar{V}\right]_{\mathrm{left}} - \left[E_{\mathrm{VBM}} - \bar{V}\right]_{\mathrm{right}} + \left[\bar{V}_{\mathrm{left}} - \bar{V}_{\mathrm{right}}\right]_{\mathrm{superlattice}}$$

The offset is positive when the valence band maximum of the left part lies above that of the right part. The bulk cell of each part is built from the interface: one period of the part's crystal along the interface normal, strained as in the interface.

The result is meaningful when:

- the interface is a periodic superlattice with no vacuum, and its two interfaces are equivalent;
- each block is at least one bulk cell thick, so that the averaged ESP is flat at its centre (the plot at the end shows it);
- the two parts differ in composition, as the blocks are found by element.

<h2 style="color:green">Usage</h2>

1. Create and save an interface, whose build metadata (`metadata.build`) holds its substrate and film, with `create_interface_with_min_strain_zsl.ipynb`: a superlattice with `INTERFACE_VACUUM = 0.0`, `INTERFACE_DISTANCE` set to the spacing of atomic planes across the interface (for example the mean of the two materials' plane spacings along the normal), and each part at least one layer thick.
1. Set the interface and calculation parameters in cells 1.2 and 1.3 below.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the VBO result.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure parameters for the interface, workflow, compute resources, and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Create materials: load an interface from the `../uploads` folder, build the bulk cells of its left and right parts from the interface's build metadata (`metadata.build`), strip the interface labels not compatible with Quantum ESPRESSO, and save all three materials to the platform tagged with the interface name.
1. Configure workflow: select application, load the VBO (Bulk Method) workflow from Standata, set model and computational parameters with a k-grid per material, and preview the workflow.
1. Configure compute: get list of clusters and create compute configuration with selected cluster, queue, and number of processors.
1. Create the job with materials and workflow configuration: assemble the job from materials, workflow, project, and compute configuration.
1. Submit the job and monitor the status: submit the job and wait for completion.
1. Retrieve results: get and display the valence band offset, the average ESP profiles, and the superlattice ESP with the two block averages.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters
Provide the INTERFACE_NAME to match the name of the structure in "uploads" folder for search. `LEFT_SIDE_PART` and `RIGHT_SIDE_PART` select the substrate or the film of the interface's build metadata (`metadata.build`).

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
INTERFACE_NAME = "Interface"  # To search for in "uploads" folder, account's materials, then Standata
RELAXED_INTERFACE_NAME = None  # To search for in "uploads" folder, account's materials, then Standata. Defaults to the interface
LEFT_SIDE_PART = InterfacePartsEnum.SUBSTRATE
RIGHT_SIDE_PART = InterfacePartsEnum.FILM
INTERFACE_SYSTEM_NAME = None  # Used as tag to group the materials. Defaults to shorthand from the loaded interface name

# 4. Workflow parameters
APPLICATION_NAME = "espresso"
WORKFLOW_SEARCH_TERM = "valence_band_offset_bulk.json"
MY_WORKFLOW_NAME = "VBO Bulk"

# 5. Compute parameters
CLUSTER_NAME = None
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.3. Set specific VBO parameters
The k-grids of the superlattice and of the two bulk cells follow from one k-point density, so all three are sampled alike.

In [ ]:
# Method parameters
PSEUDOPOTENTIAL_TYPE = "us"  # "us" (ultrasoft), "nc" (norm-conserving), "paw"
FUNCTIONAL = "pbe"  # for gga: "pbe", "pbesol"; for lda: "pz"
MODEL_SUBTYPE = "gga"

# K-grid and k-path
KPOINT_DENSITY = 3.5  # points per Å⁻¹ of reciprocal length
KPATH = None  # e.g. [{"point": "Γ", "steps": 20}, {"point": "X", "steps": 20}]

# Set SCF diagonalization and mixing in QE input
ADDITIONAL_PARAMETERS = {"electrons":
    {
        "diagonalization": "david",  # "david" or "cg"
        "mixing_beta": 0.3
    }
}

# Energy cutoffs
ECUTWFC = 40
ECUTRHO = 200

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create materials
### 3.1. Load interface from local folder

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import load_material
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

interface = load_material(client, FOLDER, INTERFACE_NAME, ACCOUNT_ID)
relaxed_interface = load_material(client, FOLDER, RELAXED_INTERFACE_NAME, ACCOUNT_ID) if RELAXED_INTERFACE_NAME else interface

visualize(relaxed_interface, repetitions=[1, 1, 1])
visualize(relaxed_interface, repetitions=[1, 1, 1], rotation="-90x")

### 3.2. Create materials from interface parts
The bulk cell of each part is built from the build metadata (`metadata.build`) of `INTERFACE_NAME`: one period of the part's crystal along the interface normal, strained as in the interface. With `RELAXED_INTERFACE_NAME` set, the bulk cells keep the unrelaxed interface's strain and height along the normal. Each bulk cell's in-plane cell can differ from the interface's. The interface labels are removed as they are not compatible with Quantum ESPRESSO. The three materials (interface, left bulk, right bulk) are named and visualized.

In [ ]:
import re
from mat3ra.made.tools.analyze.interface_material import create_interface_bulk_cell

interface_shorthand_match = re.match(r"^(.+?)\s", interface.name)
interface_system_name = INTERFACE_SYSTEM_NAME or (
    interface_shorthand_match.group(1) if interface_shorthand_match else INTERFACE_NAME
)
left_material = create_interface_bulk_cell(interface, LEFT_SIDE_PART)
right_material = create_interface_bulk_cell(interface, RIGHT_SIDE_PART)
interface_material = relaxed_interface.clone()

interface_material.basis.set_labels_from_list([])

left_material.name = f"{interface_material.name} Left Bulk"
right_material.name = f"{interface_material.name} Right Bulk"

materials_by_role = {"interface": interface_material, "left": left_material, "right": right_material}
for role, material in materials_by_role.items():
    print(f"  {role}: {material.name}")
visualize(list(materials_by_role.values()), repetitions=[1, 1, 1], rotation="-90x")

### 3.3. Save materials

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = {}
for role, material in materials_by_role.items():
    saved_response = get_or_create_material(client, material, ACCOUNT_ID)
    saved = Material.create(saved_response)
    tags = sorted({*(saved_response.get("tags") or []), interface_system_name})
    client.materials.update(saved.id, {"tags": tags})
    saved_materials[role] = saved
    print(f"  {role}: {saved.name} ({saved.id}) | tags={tags}")

## 4. Configure workflow
### 4.1. Select application

In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Load workflow from Standata and preview it

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)

workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

### 4.3. Set model and its parameters (physics)

In [ ]:
from mat3ra.mode import ModelFactory
from mat3ra.standata.model_tree import ModelTreeStandata

model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)

for subworkflow in workflow.subworkflows:
    if subworkflow.application.name == APPLICATION_NAME:
        subworkflow.model = model

### 4.4. Modify method (computational parameters): k-grid per material, k-path, cutoffs, diagonalization, and mixing
Each Quantum ESPRESSO subworkflow runs on the material its "Set Material Index" unit selects, the superlattice when it has none, and gets the k-grid of that material at `KPOINT_DENSITY`.

In [ ]:
from mat3ra.wode.context.providers import PointsGridDataProvider, PointsPathDataProvider
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, kgrid_from_density, patch_workflow_qe_input

scf_unit_names = ["pw_scf", "pw_bands"]
materials = list(materials_by_role.values())

for subworkflow in workflow.subworkflows:
    if subworkflow.application.name != APPLICATION_NAME:
        continue
    material_index = int(next((unit.value for unit in subworkflow.units if unit.name.startswith("Set Material Index")), 0))
    material = materials[material_index]
    kgrid = kgrid_from_density(material, KPOINT_DENSITY)
    unit = subworkflow.get_unit_by_name(name="pw_scf")
    unit.add_context(PointsGridDataProvider(material=material, dimensions=kgrid, isEdited=True).get_context_item_data())
    subworkflow.set_unit(unit)
    print(f"  {subworkflow.name}: {material.name}, k-grid {kgrid}")

if KPATH is not None:
    for subworkflow in workflow.subworkflows:
        if unit := subworkflow.get_unit_by_name(name="pw_bands"):
            unit.add_context(PointsPathDataProvider(path=KPATH, isEdited=True).get_context_item_data())
            subworkflow.set_unit(unit)

if ECUTWFC is not None:
    for unit_name in scf_unit_names:
        apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=unit_name)

if ADDITIONAL_PARAMETERS:
    patch_workflow_qe_input(workflow, ADDITIONAL_PARAMETERS, unit_names=scf_unit_names)

### 4.5. Preview final workflow

In [ ]:
visualize_workflow(workflow)

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job
### 6.1. Create job

In [ ]:
from mat3ra.notebooks_utils.job import create_job
from mat3ra.utils.namespace import dict_to_namespace_recursive
from mat3ra.notebooks_utils.ui import display_JSON

materials = list(saved_materials.values())

job_name = f"{MY_WORKFLOW_NAME} {interface_system_name} {timestamp}"
workflow.name = job_name

print(f"Materials: {[m.id for m in materials]}")
print(f"Project: {project_id}")

job_response = create_job(
    api_client=client,
    materials=materials,
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job = dict_to_namespace_recursive(job_response)
job_id = job._id
print("✅ Job created successfully!")
print(f"Job ID: {job_id}")

display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve and visualize results
### 8.1. Valence Band Offset and average ESP profiles

In [ ]:
from mat3ra.prode import PropertyName
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

job_data = client.jobs.get(job_id)
job_scope = {key: value for item in job_data["scopeTrack"] for key, value in item["scope"]["global"].items()}

vbo_value = client.properties.get_for_job(
    job_id,
    property_name=PropertyName.scalar.valence_band_offset.value,
)[0]
print(f"Valence Band Offset (VBO): {float(vbo_value['value']):.3f} eV")

avg_esp_results = {}
for subworkflow in job_data["workflow"]["subworkflows"]:
    for unit in subworkflow["units"]:
        if "average_potential_profile" in [result["name"] for result in unit.get("results", [])]:
            avg_esp_results[subworkflow["name"]] = client.properties.get_for_job(
                job_id,
                property_name="average_potential_profile",
                unit_id=unit["flowchartId"],
            )[0]

for subworkflow_name, avg_esp_data in avg_esp_results.items():
    visualize_properties(avg_esp_data, title=subworkflow_name)

### 8.2. Plot the superlattice ESP profile with the block averages
The workflow averages the ESP of the superlattice over one bulk cell at the centre of each block. A block is bulk-like where the macroscopic average is flat on its block average.

In [ ]:
from mat3ra.notebooks_utils.plot import configure_matplotlib_renderer

configure_matplotlib_renderer()
import matplotlib.pyplot as plt

interface_profile = avg_esp_results["Avg ESP (Interface)"]
plt.figure(figsize=(10, 6))
plt.plot(interface_profile["xDataArray"], interface_profile["yDataSeries"][1], label="Macroscopic Average Potential", linewidth=2)
for block_name, block_average, color in zip(["Left", "Right"], job_scope["AVG_ESP_INTERFACE"], ["gold", "blue"]):
    plt.axhline(block_average, color=color, linestyle=":", linewidth=2, label=f"{block_name} block average ESP = {block_average:.3f} eV")
plt.xlabel("z-coordinate (A)", fontsize=12)
plt.ylabel("Macroscopic Average Potential (eV)", fontsize=12)
plt.title(f"Valence band offset: {float(vbo_value['value']):.3f} eV", fontsize=14, fontweight="bold")
plt.legend(loc="best", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## References

[1] C. G. Van de Walle and R. M. Martin, "Theoretical calculations of heterojunction discontinuities in the Si/Ge system", Phys. Rev. B 34, 5621 (1986). [DOI: 10.1103/PhysRevB.34.5621](https://doi.org/10.1103/PhysRevB.34.5621)

[2] A. Baldereschi, S. Baroni and R. Resta, "Band offsets in lattice-matched heterojunctions: a model and first-principles calculations for GaAs/AlAs", Phys. Rev. Lett. 61, 734 (1988). [DOI: 10.1103/PhysRevLett.61.734](https://doi.org/10.1103/PhysRevLett.61.734)